In [16]:
import cv2
from ultralytics import YOLO
import math

def calcular_angulo(A, B, C):
    """
    Calcula el ángulo en el punto B.
    A, B, C son tuplas (x, y)
    """
    radianes = math.atan2(C[1] - B[1], C[0] - B[0]) - \
               math.atan2(A[1] - B[1], A[0] - B[0])
    angulo = abs(radianes * 180.0 / math.pi)
    if angulo > 180.0:
        angulo = 360 - angulo
    return angulo

In [17]:
model = YOLO("yolo11n-pose.pt")

In [18]:
# Ruta al video (CAMBIA ESTO según tu video)
video_path = "entrenamiento.mp4"  
cap = cv2.VideoCapture(video_path)

# Configurar el video de salida
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('video_procesado.mp4', fourcc, fps, (width, height))

# ===== CONFIGURACIÓN DEL EJERCICIO =====
# Para SENTADILLAS: usar rodilla (cadera-rodilla-tobillo)
PUNTO_A = 11  # Cadera izquierda (para sentadillas)
PUNTO_B = 13  # Rodilla izquierda (vértice del ángulo)
PUNTO_C = 15  # Tobillo izquierdo

# Para FLEXIONES: descomentar estas líneas
# PUNTO_A = 5   # Hombro izquierdo
# PUNTO_B = 7   # Codo izquierdo (vértice)
# PUNTO_C = 9   # Muñeca izquierda

# Umbrales de ángulo (ajústables según el ejercicio)
UMBRAL_ARRIBA = 150  # Ángulo cuando está "arriba" (extendido)
UMBRAL_ABAJO = 100   # Ángulo cuando está "abajo" (flexionado)

# Variables para contar repeticiones
contador = 0
estado = None
frame_count = 0
angulos_registrados = []

print("🎬 Procesando video...")

while cap.isOpened():
    ret, frame = cap.read()
    
    if not ret:
        break
    
    frame_count += 1
    results = model(frame, verbose=False)
    
    persona_detectada = False
    
    for r in results:
        # VALIDAR QUE HAY DETECCIONES
        if r.keypoints is None or len(r.keypoints.xy) == 0:
            continue
            
        kpts = r.keypoints.xy[0]
        
        # VALIDAR QUE SE DETECTARON LOS 17 PUNTOS
        if kpts.shape[0] < 17:
            continue
        
        persona_detectada = True
        
        # Extraer puntos configurables
        punto_a = (int(kpts[PUNTO_A][0]), int(kpts[PUNTO_A][1]))
        punto_b = (int(kpts[PUNTO_B][0]), int(kpts[PUNTO_B][1]))  # Vértice
        punto_c = (int(kpts[PUNTO_C][0]), int(kpts[PUNTO_C][1]))
        
        # Calcular ángulo
        angulo = calcular_angulo(punto_a, punto_b, punto_c)
        angulos_registrados.append(angulo)
        
        # Lógica de conteo
        if angulo > UMBRAL_ARRIBA:
            if estado == "abajo":
                contador += 1
                print(f"✅ Repetición {contador} en frame {frame_count}")
            estado = "arriba"
        elif angulo < UMBRAL_ABAJO:
            estado = "abajo"
        
        # Dibujar líneas entre articulaciones
        cv2.line(frame, punto_a, punto_b, (255, 255, 255), 3)
        cv2.line(frame, punto_b, punto_c, (255, 255, 255), 3)
        
        # Dibujar círculos en las articulaciones
        cv2.circle(frame, punto_a, 8, (0, 255, 0), -1)
        cv2.circle(frame, punto_b, 8, (0, 0, 255), -1)
        cv2.circle(frame, punto_c, 8, (0, 255, 0), -1)
        
        # Escribir el ángulo
        cv2.putText(frame, f"Angulo: {int(angulo)}", 
                   (punto_b[0] - 50, punto_b[1] - 20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        
        # Escribir el contador de repeticiones
        cv2.putText(frame, f"Repeticiones: {contador}", 
                   (50, 50),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
        
        # Escribir el estado
        if estado:
            cv2.putText(frame, f"Estado: {estado}", 
                       (50, 100),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
    
    # Si no se detectó persona, mostrar advertencia
    if not persona_detectada:
        cv2.putText(frame, "No se detecta persona", 
                   (50, 150),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # Guardar el frame procesado
    out.write(frame)
    
    # Mostrar progreso cada 30 frames
    if frame_count % 30 == 0:
        print(f"📊 Procesados {frame_count} frames, Repeticiones: {contador}")

cap.release()
out.release()

print(f"\n✅ Video procesado guardado como 'video_procesado.mp4'")
print(f"🏋️ Total de repeticiones contadas: {contador}")

if angulos_registrados:
    print(f"📐 Ángulos detectados - Min: {min(angulos_registrados):.1f}°, Max: {max(angulos_registrados):.1f}°")
    print(f"📐 Ángulo promedio: {sum(angulos_registrados)/len(angulos_registrados):.1f}°")

🎬 Procesando video...
📊 Procesados 30 frames, Repeticiones: 0
📊 Procesados 60 frames, Repeticiones: 0
📊 Procesados 90 frames, Repeticiones: 0
📊 Procesados 120 frames, Repeticiones: 0
📊 Procesados 150 frames, Repeticiones: 0
📊 Procesados 180 frames, Repeticiones: 0
📊 Procesados 210 frames, Repeticiones: 0
📊 Procesados 240 frames, Repeticiones: 0
📊 Procesados 270 frames, Repeticiones: 0
📊 Procesados 300 frames, Repeticiones: 0
📊 Procesados 330 frames, Repeticiones: 0
📊 Procesados 360 frames, Repeticiones: 0
📊 Procesados 390 frames, Repeticiones: 0
📊 Procesados 420 frames, Repeticiones: 0
📊 Procesados 450 frames, Repeticiones: 0
📊 Procesados 480 frames, Repeticiones: 0
📊 Procesados 510 frames, Repeticiones: 0
📊 Procesados 540 frames, Repeticiones: 0
✅ Repetición 1 en frame 564
📊 Procesados 570 frames, Repeticiones: 1
📊 Procesados 600 frames, Repeticiones: 1
📊 Procesados 630 frames, Repeticiones: 1
📊 Procesados 660 frames, Repeticiones: 1
📊 Procesados 690 frames, Repeticiones: 1
📊 Procesad

In [ ]:
# Ruta al video (CAMBIA ESTO según tu video)
video_path = "entrenamiento1.mp4"  
cap = cv2.VideoCapture(video_path)

# Configurar el video de salida
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('video_procesado1.mp4', fourcc, fps, (width, height))

# ===== CONFIGURACIÓN DEL EJERCICIO =====
# Para SENTADILLAS: usar rodilla (cadera-rodilla-tobillo)
PUNTO_A = 11  # Cadera izquierda (para sentadillas)
PUNTO_B = 13  # Rodilla izquierda (vértice del ángulo)
PUNTO_C = 15  # Tobillo izquierdo

# Para FLEXIONES: descomentar estas líneas
# PUNTO_A = 5   # Hombro izquierdo
# PUNTO_B = 7   # Codo izquierdo (vértice)
# PUNTO_C = 9   # Muñeca izquierda

# Umbrales de ángulo (ajústables según el ejercicio)
UMBRAL_ARRIBA = 150  # Ángulo cuando está "arriba" (extendido)
UMBRAL_ABAJO = 100   # Ángulo cuando está "abajo" (flexionado)

# Variables para contar repeticiones
contador = 0
estado = None
frame_count = 0
angulos_registrados = []

print("🎬 Procesando video...")

while cap.isOpened():
    ret, frame = cap.read()
    
    if not ret:
        break
    
    frame_count += 1
    results = model(frame, verbose=False)
    
    persona_detectada = False
    
    for r in results:
        # VALIDAR QUE HAY DETECCIONES
        if r.keypoints is None or len(r.keypoints.xy) == 0:
            continue
            
        kpts = r.keypoints.xy[0]
        
        # VALIDAR QUE SE DETECTARON LOS 17 PUNTOS
        if kpts.shape[0] < 17:
            continue
        
        persona_detectada = True
        
        # Extraer puntos configurables
        punto_a = (int(kpts[PUNTO_A][0]), int(kpts[PUNTO_A][1]))
        punto_b = (int(kpts[PUNTO_B][0]), int(kpts[PUNTO_B][1]))  # Vértice
        punto_c = (int(kpts[PUNTO_C][0]), int(kpts[PUNTO_C][1]))
        
        # Calcular ángulo
        angulo = calcular_angulo(punto_a, punto_b, punto_c)
        angulos_registrados.append(angulo)
        
        # Lógica de conteo
        if angulo > UMBRAL_ARRIBA:
            if estado == "abajo":
                contador += 1
                print(f"✅ Repetición {contador} en frame {frame_count}")
            estado = "arriba"
        elif angulo < UMBRAL_ABAJO:
            estado = "abajo"
        
        # Dibujar líneas entre articulaciones
        cv2.line(frame, punto_a, punto_b, (255, 255, 255), 3)
        cv2.line(frame, punto_b, punto_c, (255, 255, 255), 3)
        
        # Dibujar círculos en las articulaciones
        cv2.circle(frame, punto_a, 8, (0, 255, 0), -1)
        cv2.circle(frame, punto_b, 8, (0, 0, 255), -1)
        cv2.circle(frame, punto_c, 8, (0, 255, 0), -1)
        
        # Escribir el ángulo
        cv2.putText(frame, f"Angulo: {int(angulo)}", 
                   (punto_b[0] - 50, punto_b[1] - 20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        
        # Escribir el contador de repeticiones
        cv2.putText(frame, f"Repeticiones: {contador}", 
                   (50, 50),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
        
        # Escribir el estado
        if estado:
            cv2.putText(frame, f"Estado: {estado}", 
                       (50, 100),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
    
    # Si no se detectó persona, mostrar advertencia
    if not persona_detectada:
        cv2.putText(frame, "No se detecta persona", 
                   (50, 150),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # Guardar el frame procesado
    out.write(frame)
    
    # Mostrar progreso cada 30 frames
    if frame_count % 30 == 0:
        print(f"📊 Procesados {frame_count} frames, Repeticiones: {contador}")

cap.release()
out.release()

print(f"\n✅ Video procesado guardado como 'video_procesado.mp4'")
print(f"🏋️ Total de repeticiones contadas: {contador}")

if angulos_registrados:
    print(f"📐 Ángulos detectados - Min: {min(angulos_registrados):.1f}°, Max: {max(angulos_registrados):.1f}°")
    print(f"📐 Ángulo promedio: {sum(angulos_registrados)/len(angulos_registrados):.1f}°")

# En este video lo que he detectado es que al estar dos personas en el video, me lo detecta mal, esta recogiendo el angulo y las repeticiones de los dos.

🎬 Procesando video...
📊 Procesados 30 frames, Repeticiones: 0
📊 Procesados 60 frames, Repeticiones: 0
📊 Procesados 90 frames, Repeticiones: 0
📊 Procesados 120 frames, Repeticiones: 0
📊 Procesados 150 frames, Repeticiones: 0
📊 Procesados 180 frames, Repeticiones: 0
📊 Procesados 210 frames, Repeticiones: 0
📊 Procesados 240 frames, Repeticiones: 0
📊 Procesados 270 frames, Repeticiones: 0
📊 Procesados 300 frames, Repeticiones: 0
📊 Procesados 330 frames, Repeticiones: 0
📊 Procesados 360 frames, Repeticiones: 0
📊 Procesados 390 frames, Repeticiones: 0
📊 Procesados 420 frames, Repeticiones: 0
📊 Procesados 450 frames, Repeticiones: 0
📊 Procesados 480 frames, Repeticiones: 0
📊 Procesados 510 frames, Repeticiones: 0
📊 Procesados 540 frames, Repeticiones: 0
📊 Procesados 570 frames, Repeticiones: 0
📊 Procesados 600 frames, Repeticiones: 0
📊 Procesados 630 frames, Repeticiones: 0
📊 Procesados 660 frames, Repeticiones: 0
📊 Procesados 690 frames, Repeticiones: 0
📊 Procesados 720 frames, Repeticiones: